# Debt & Covenant Breach Simulator

**Purpose**  
- Quantify the probability that debt and covenants destroy equity value before the operating thesis has time to materialise

## 1. Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 2. Economic Baseline Assumptions

In [ ]:
MONTHS = 60
N_SIMS = 10_000
INITIAL_CASH = 40_000_000
MONTHLY_FIXED_COST = 6_000_000
MAX_MONTHLY_REVENUE = 26_000_000
GROSS_MARGIN_FULL_YIELD = 0.34

## 3. Operating Dynamics (Yield + Shocks)

In [ ]:
INITIAL_YIELD = 0.92
YIELD_DRIFT = -0.002
YIELD_VOL = 0.03

# Operational shock
ANNUAL_OP_SHOCK_RATE = 0.5
MONTHLY_OP_SHOCK_PROB = ANNUAL_OP_SHOCK_RATE / 12
OP_SHOCK_YIELD_DROP = 0.25
OP_RECOVERY_MONTHS = 6

# Customer loss
ANNUAL_CUSTOMER_LOSS_RATE = 0.25
MONTHLY_CUSTOMER_LOSS_PROB = ANNUAL_CUSTOMER_LOSS_RATE / 12
CUSTOMER_REVENUE_LOSS = 0.35
CUSTOMER_RECOVERY_MONTHS = 9

## 4. Debt Structure Assumptions

In [ ]:
INITIAL_NET_DEBT = 160_000_000
INTEREST_RATE = 0.075
MONTHLY_INTEREST = INITIAL_NET_DEBT * INTEREST_RATE / 12
PRINCIPAL_AMORT = 1_000_000

## 5. Covenant Definitions

In [ ]:
MAX_LEVERAGE = 4.5      # Net Debt / EBITDA
MIN_INTEREST_COVER = 2.0  # EBITDA / Interest

## 6. Single‑Path Simulation Engine

In [ ]:
def simulate_path():
    cash = INITIAL_CASH
    yld = INITIAL_YIELD
    net_debt = INITIAL_NET_DEBT

    op_active = False
    op_residual = 0.0

    cust_active = False
    cust_months = 0
    cust_factor = 1.0

    for t in range(MONTHS):

        # Shock arrivals
        if (not op_active) and (np.random.rand() < MONTHLY_OP_SHOCK_PROB):
            op_active = True
            op_residual = OP_SHOCK_YIELD_DROP

        if (not cust_active) and (np.random.rand() < MONTHLY_CUSTOMER_LOSS_PROB):
            cust_active = True
            cust_months = CUSTOMER_RECOVERY_MONTHS
            cust_factor = 1 - CUSTOMER_REVENUE_LOSS

        # Shock mechanics
        if op_active:
            yld *= (1 - op_residual)
            op_residual *= (1 - 1 / OP_RECOVERY_MONTHS)
            if op_residual < 0.01:
                op_active = False

        if cust_active:
            cust_months -= 1
            if cust_months <= 0:
                cust_active = False
                cust_factor = 1.0

        # Baseline drift
        yld *= np.exp(np.random.normal(YIELD_DRIFT, YIELD_VOL))
        yld = np.clip(yld, 0.0, 1.0)

        # Economics
        revenue = MAX_MONTHLY_REVENUE * yld * cust_factor
        ebitda = revenue * GROSS_MARGIN_FULL_YIELD - MONTHLY_FIXED_COST / 2
        interest = net_debt * INTEREST_RATE / 12
        cash += ebitda - interest - PRINCIPAL_AMORT
        net_debt -= PRINCIPAL_AMORT

        # Covenants
        leverage = net_debt / max(ebitda * 12, 1)
        interest_cover = ebitda / max(interest, 1)

        if leverage > MAX_LEVERAGE or interest_cover < MIN_INTEREST_COVER:
            return 1, t + 1  # covenant breach

        if cash <= 0:
            return 2, t + 1  # cash ruin

    return 0, MONTHS

## 7. Monte Carlo Simulation

In [ ]:
outcome = []
time_to_event = []

for _ in range(N_SIMS):
    r, t = simulate_path()
    outcome.append(r)
    time_to_event.append(t)

outcome = np.array(outcome)
time_to_event = np.array(time_to_event)

## 8. Core Risk Outputs

In [ ]:
prob_breach = np.mean(outcome == 1)
prob_ruin = np.mean(outcome == 2)

prob_breach, prob_ruin

## 9. Visualization

In [ ]:
plt.hist(time_to_event[outcome > 0], bins=30)
plt.title("Time to Covenant Breach or Cash Ruin")
plt.xlabel("Months")
plt.ylabel("Frequency")
plt.show()

## 10. Deal Interpretation


A significant share of downside paths fail due to **capital structure constraints**, not operating collapse.

The analysis highlights scenarios where the operating thesis remains viable but equity value is destroyed by:
- Maintenance covenant breaches
- Insufficient headroom for recovery
- Premature lender intervention

This model supports debt sizing, covenant negotiation, and equity buffer calibration.
